# F1 video car detection

First 10 seconds, sample one frame per second by default (~10 API calls), run only car detection, then save detection stubs to CSV. No team classification or driver lookup.

In [ ]:
from pathlib import Path
import csv
from collections import defaultdict
import cv2

from config import settings
from detection import get_detections, get_detection_info, get_inference_time
from roboflow_client import run_rf_detr

from annotate import (
    annotate_car_bbox,
    annotate_team_colour,
    annotate_team_label,
    annotate_driver_label,
)

FILENAME = "race-clip.mp4"
VIDEO_PATH = Path(settings.INPUTS_DIR) / FILENAME 
CLIP_SECONDS = 10.0
SAMPLE_EVERY_SECONDS = 0.25 # How many times the API will be queried per second. 
OUTPUT_DIR = Path(settings.OUTPUTS_DIR) / VIDEO_PATH.stem / "detection_stubs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / f"{VIDEO_PATH.stem}_detections.csv"

## 1. Video metrics

`total_frames` and `source_duration_s` describe the whole source video; `frames_in_window` describes the first 10 seconds. Trying to understand some basic metrcics before exhausing api credits.

In [3]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

video_metrics = {
    "width_px": width,
    "height_px": height,
    "fps": fps,
    "total_frames": total_frames,
    "source_duration_s": total_frames / fps,
    "processing_window_s": CLIP_SECONDS,
    "frames_in_window": min(total_frames, int(CLIP_SECONDS * fps)),
    "sample_every_seconds": SAMPLE_EVERY_SECONDS
}

print(video_metrics)

{'width_px': 1280, 'height_px': 720, 'fps': 30.0, 'total_frames': 187, 'source_duration_s': 6.233333333333333, 'processing_window_s': 10.0, 'frames_in_window': 187, 'sample_every_seconds': 0.5}


## 2. Sample frames and detect cars

Frames are decoded locally, but only selected frames are sent to Roboflow. OpenCV BGR frames are converted to RGB. The CSV stores center/size and corner coordinates.

In [4]:
''' CSV Variables. '''

rows, api_calls, total_inference_s = [], 0, 0.0
next_sample_s, frame_index = 0.0, 0

fields = [
    "frame_index",
    "timestamp_s",
    "detection_index",
    "class_label",
    "class_id",
    "confidence",
    "center_x",
    "center_y",
    "width",
    "height",
    "x1",
    "y1",
    "x2",
    "y2",
    "inference_time_s"
]

In [10]:
'''  Car Detection Loop. '''

try:

    while True:

        ok, frame_bgr = cap.read()

        if not ok:
            break

        timestamp_s = frame_index / fps

        if timestamp_s >= CLIP_SECONDS:
            break

        if timestamp_s + 0.5 / fps >= next_sample_s:

            rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            
            result = run_rf_detr(
                frame=rgb,
                workspace_id=settings.WORKSPACE_ID,
                model_id=settings.CAR_DETECTOR_ID,
                api_key=settings.ROBOFLOW_API_KEY
            )

            api_calls += 1
            inference_s = get_inference_time(result)
            total_inference_s += inference_s

            for detection_id, pred in enumerate(get_detections(result), start=1):

                item = get_detection_info(detection_id, result)
                bbox = item["bbox"]

                rows.append({
                    "frame_index": frame_index,
                    "timestamp_s": round(timestamp_s, 3),
                    "detection_index": detection_id,
                    "class_label": item["class_label"],
                    "class_id": item["class_id"],
                    "confidence": pred.get("confidence", ""),
                    "center_x": pred.get("x", ""),
                    "center_y": pred.get("y", ""),
                    "width": pred.get("width", ""),
                    "height": pred.get("height", ""),
                    "x1": bbox["x1"], 
                    "y1": bbox["y1"],
                    "x2": bbox["x2"],
                    "y2": bbox["y2"],
                    "inference_time_s": inference_s
                })

            next_sample_s += SAMPLE_EVERY_SECONDS
            
        frame_index += 1

finally:

    cap.release()

with CSV_PATH.open("w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

print({
    "api_calls": api_calls,
    "detections": len(rows),
    "inference_time_s": round(total_inference_s, 3),
    "csv": str(CSV_PATH)
})

{'api_calls': 13, 'detections': 19, 'inference_time_s': 0.408, 'csv': 'd:\\general_programming\\python\\f1\\outputs\\race-clip\\detection_stubs\\race-clip_detections.csv'}


## 3. Review

Empty detections count as API calls but produce no CSV rows.

In [11]:
rows[:10]

[{'frame_index': 0,
  'timestamp_s': 0.0,
  'detection_index': 1,
  'class_label': 'car',
  'class_id': 0,
  'confidence': 0.7981130480766296,
  'center_x': 570.5,
  'center_y': 392.5,
  'width': 687.0,
  'height': 301.0,
  'x1': 227,
  'y1': 242,
  'x2': 914,
  'y2': 543,
  'inference_time_s': 0.13989712577313185},
 {'frame_index': 0,
  'timestamp_s': 0.0,
  'detection_index': 2,
  'class_label': 'car',
  'class_id': 0,
  'confidence': 0.7474663257598877,
  'center_x': 1083.5,
  'center_y': 338.0,
  'width': 393.0,
  'height': 242.0,
  'x1': 887,
  'y1': 217,
  'x2': 1280,
  'y2': 459,
  'inference_time_s': 0.13989712577313185},
 {'frame_index': 15,
  'timestamp_s': 0.5,
  'detection_index': 1,
  'class_label': 'car',
  'class_id': 0,
  'confidence': 0.9225035309791565,
  'center_x': 458.0,
  'center_y': 398.0,
  'width': 696.0,
  'height': 256.0,
  'x1': 110,
  'y1': 270,
  'x2': 806,
  'y2': 526,
  'inference_time_s': 0.024727679789066315},
 {'frame_index': 15,
  'timestamp_s': 0.5,

In [5]:
def render_annotated_video(video_path, csv_path, output_path):

    # Group detection rows by the video frame they belong to.
    detections_by_frame = defaultdict(list)

    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            frame_index = int(row["frame_index"])
            detections_by_frame[frame_index].append(row)

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(
        str(output_path), fourcc, fps, (width, height)
    )

    if not writer.isOpened():
        cap.release()
        raise ValueError(f"Could not create output video: {output_path}")

    frame_index = 0

    try:

        while True:

            success, frame = cap.read()

            if not success:
                break

            annotated_frame = frame

            for row in detections_by_frame.get(frame_index, []):

                # Your annotation functions expect (x1, y1, x2, y2).
                bbox = tuple(
                    int(round(float(row[key])))
                    for key in ("x1", "y1", "x2", "y2")
                )

                # The current detection CSV has no team or driver fields,
                # so it will draw the box in green for now.
                team_name = row.get("team_name") or row.get("team")
                driver_name = row.get("driver_name") or row.get("driver")

                colour = (
                    annotate_team_colour(team_name)
                    if team_name
                    else (0, 255, 0)
                )

                annotated_frame = annotate_car_bbox(
                    annotated_frame, bbox, colour
                )

                if team_name:
                    annotated_frame = annotate_team_label(
                        annotated_frame, bbox, team_name
                    )

                if driver_name:
                    annotated_frame = annotate_driver_label(
                        annotated_frame, bbox, driver_name
                    )

            writer.write(annotated_frame)
            frame_index += 1

    finally:

        cap.release()
        writer.release()

    return output_path, frame_index

In [6]:
ANNOTATED_VIDEO_PATH = OUTPUT_DIR / f"{VIDEO_PATH.stem}_annotated.mp4"

output_path, frames_written = render_annotated_video(
    VIDEO_PATH,
    CSV_PATH,
    ANNOTATED_VIDEO_PATH,
)

print(f"Saved {frames_written} frames to {output_path}")

Saved 187 frames to d:\general_programming\python\f1\outputs\race-clip\detection_stubs\race-clip_annotated.mp4


## 3. Tracking

ByteTrack assigns IDs to the sampled detections. The render step interpolates boxes between consecutive observations of the same confirmed track, so annotations stay visible between detector calls.

In [ ]:
from trackers import ByteTrackTracker
import pandas as pd
import numpy as np
import supervision as sv

# Tracking variables.
TRACK_ACTIVATION_THRESHOLD = 0.25
LOST_TRACK_BUFFER = 30
MINIMUM_IOU_THRESHOLD = 0.1

rows = (
    pd.read_csv(CSV_PATH)
    .sort_values(["frame_index", "detection_index"])
    .reset_index(drop=True)
)
rows["frame_index"] = pd.to_numeric(rows["frame_index"]).astype(int)
rows["_row_id"] = np.arange(len(rows), dtype=np.int64)
track_ids = np.full(len(rows), -1, dtype=np.int64)

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise ValueError(f"Could not open video: {VIDEO_PATH}")

fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
cap.release()
if fps <= 0:
    raise ValueError(f"Could not read a valid FPS value from: {VIDEO_PATH}")

tracker = ByteTrackTracker(
    frame_rate=fps,
    track_activation_threshold=TRACK_ACTIVATION_THRESHOLD,
    lost_track_buffer=LOST_TRACK_BUFFER,
    minimum_iou_threshold=MINIMUM_IOU_THRESHOLD,
    minimum_consecutive_frames=1,
)

for _, group in rows.groupby("frame_index", sort=True):
    source_rows = group["_row_id"].to_numpy(dtype=np.int64)
    confidences = (
        pd.to_numeric(group["confidence"], errors="coerce")
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    detections = sv.Detections(
        xyxy=group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=np.float32),
        confidence=confidences,
        class_id=group["class_id"].fillna(-1).to_numpy(dtype=np.int32),
        data={"source_row": source_rows},
    )

    timestamp_s = float(group["timestamp_s"].iloc[0])
    tracked = tracker.update(detections, timestamp=timestamp_s)

    if len(tracked) and tracked.tracker_id is not None:
        tracked_rows = tracked.data["source_row"].astype(np.int64)
        track_ids[tracked_rows] = tracked.tracker_id.astype(np.int64)

rows["track_id"] = track_ids
rows["box_source"] = "detector"
rows = rows.drop(columns="_row_id")

TRACKED_CSV_PATH = CSV_PATH.parent / "tracked_detections.csv"
rows.to_csv(TRACKED_CSV_PATH, index=False)

print(f"Saved tracked detections: {TRACKED_CSV_PATH}")
print(rows[["frame_index", "timestamp_s", "detection_index", "track_id"]])

## Render continuous tracks

Interpolate between nearby observations for each confirmed track. Single-observation and unassigned detections are omitted from the video to avoid one-frame flashes.

In [ ]:
MAX_INTERPOLATION_GAP_SECONDS = SAMPLE_EVERY_SECONDS * 2
MAX_INTERPOLATION_GAP_FRAMES = max(1, int(round(MAX_INTERPOLATION_GAP_SECONDS * fps)))

tracked_detections = pd.read_csv(TRACKED_CSV_PATH)
tracked_detections["frame_index"] = pd.to_numeric(
    tracked_detections["frame_index"]
).astype(int)
tracked_detections["track_id"] = pd.to_numeric(
    tracked_detections["track_id"], errors="coerce"
).fillna(-1).astype(int)

# Build one box per track per frame, interpolating only short gaps between
# consecutive confirmed detector observations.
boxes_by_track_frame = {}

confirmed = tracked_detections[tracked_detections["track_id"] >= 0]
for track_id, track_rows in confirmed.groupby("track_id", sort=True):
    track_rows = track_rows.sort_values("frame_index").drop_duplicates(
        "frame_index",
        keep="last",
    )
    observations = track_rows.to_dict("records")

    for left, right in zip(observations, observations[1:]):
        start_frame = int(left["frame_index"])
        end_frame = int(right["frame_index"])
        gap = end_frame - start_frame

        if gap <= 0 or gap > MAX_INTERPOLATION_GAP_FRAMES:
            continue

        start_box = np.array(
            [left["x1"], left["y1"], left["x2"], left["y2"]],
            dtype=np.float64,
        )
        end_box = np.array(
            [right["x1"], right["y1"], right["x2"], right["y2"]],
            dtype=np.float64,
        )

        for frame_index in range(start_frame, end_frame + 1):
            if frame_index == start_frame:
                box = start_box
                box_source = "detector"
            elif frame_index == end_frame:
                box = end_box
                box_source = "detector"
            else:
                fraction = (frame_index - start_frame) / gap
                box = start_box + fraction * (end_box - start_box)
                box_source = "interpolated"

            boxes_by_track_frame[(int(track_id), frame_index)] = {
                "frame_index": frame_index,
                "timestamp_s": round(frame_index / fps, 3),
                "track_id": int(track_id),
                "x1": int(round(box[0])),
                "y1": int(round(box[1])),
                "x2": int(round(box[2])),
                "y2": int(round(box[3])),
                "box_source": box_source,
            }

tracked_frames = pd.DataFrame(
    boxes_by_track_frame.values(),
    columns=[
        "frame_index",
        "timestamp_s",
        "track_id",
        "x1",
        "y1",
        "x2",
        "y2",
        "box_source",
    ],
).sort_values(["frame_index", "track_id"])

TRACKED_FRAMES_CSV_PATH = CSV_PATH.parent / "tracked_frames.csv"
tracked_frames.to_csv(TRACKED_FRAMES_CSV_PATH, index=False)

TRACKED_VIDEO_PATH = OUTPUT_DIR / f"{VIDEO_PATH.stem}_tracked.mp4"
rows_by_frame = {
    int(frame_index): group.to_dict("records")
    for frame_index, group in tracked_frames.groupby("frame_index", sort=True)
}

cap = cv2.VideoCapture(str(VIDEO_PATH))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frames_to_render = min(total_frames, int(np.ceil(CLIP_SECONDS * fps)))

writer = cv2.VideoWriter(
    str(TRACKED_VIDEO_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

if not writer.isOpened():
    cap.release()
    raise ValueError(f"Could not create video: {TRACKED_VIDEO_PATH}")

palette = [
    (0, 255, 0),
    (255, 100, 0),
    (0, 180, 255),
    (255, 0, 200),
]

frame_index = 0
try:
    while frame_index < frames_to_render:
        ok, frame = cap.read()
        if not ok:
            break

        annotated = frame

        for row in rows_by_frame.get(frame_index, []):
            bbox = tuple(
                int(row[key])
                for key in ("x1", "y1", "x2", "y2")
            )
            track_id = int(row["track_id"])
            colour = palette[track_id % len(palette)]

            annotated = annotate_car_bbox(annotated, bbox, colour)
            cv2.putText(
                annotated,
                f"ID {track_id}",
                (bbox[0], max(18, bbox[1] - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                colour,
                2,
                cv2.LINE_AA,
            )

        writer.write(annotated)
        frame_index += 1
finally:
    cap.release()
    writer.release()

print(f"Saved {len(tracked_frames)} per-frame boxes: {TRACKED_FRAMES_CSV_PATH}")
print(f"Saved {frame_index} video frames: {TRACKED_VIDEO_PATH}")